# skull stripping

In [7]:
import os
import nibabel as nib
import numpy as np
from nibabel.processing import resample_from_to

def skull_strip_images_resample(image_folder, mask_root_folder, output_folder, mask_subpath='mri/mask.mgz'):
    """
    Apply brain masks to MRI images using a fixed mask subpath derived from each image's name,
    automatically resampling masks to match image dimensions.

    Parameters:
        image_folder (str): Folder containing MRI images (.nii or .nii.gz)
        mask_root_folder (str): Root folder containing subject folders for masks
        output_folder (str): Folder to save skull-stripped images
        mask_subpath (str): Relative path to mask inside each subject folder
    """
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    for img_file in os.listdir(image_folder):
        if not img_file.endswith(('.nii', '.nii.gz')):
            continue

        img_name = os.path.splitext(img_file)[0]  # strip .nii or .nii.gz
        img_path = os.path.join(image_folder, img_file)

        # Construct mask path using the image name as the folder name
        mask_path = os.path.join(mask_root_folder, img_name, mask_subpath)
        if not os.path.exists(mask_path):
            print(f"Mask not found for {img_file}, skipping.")
            continue

        # Load image and mask
        img_nii = nib.load(img_path)
        mask_nii = nib.load(mask_path)

        # Resample mask to match image dimensions
        mask_resampled_nii = resample_from_to(mask_nii, img_nii, order=0)  # nearest-neighbor
        mask_data = (mask_resampled_nii.get_fdata() > 0).astype(np.float32)

        # Apply mask
        img_data = img_nii.get_fdata()
        skull_stripped_data = img_data * mask_data

        # Save result
        output_path = os.path.join(output_folder, img_file)
        nib.save(nib.Nifti1Image(skull_stripped_data, img_nii.affine, img_nii.header), output_path)
        print(f"Saved skull-stripped image: {output_path}")


In [9]:
skull_strip_images_resample(
    image_folder='MRIs_IMAGES_FILTERED_NOMASKS',
    mask_root_folder='FASTSURFER_MASKS',
    output_folder='MRIs_SPLIT_CLASSIFICATION'
)


Saved skull-stripped image: MRIs_SPLIT_CLASSIFICATION/ADNI_007_S_0068_MR_MPR-R__GradWarp__B1_Correction__N3__Scaled_Br_20100601172719474_S78802_I176204.nii
Saved skull-stripped image: MRIs_SPLIT_CLASSIFICATION/ADNI_036_S_0945_MR_MPR-R__GradWarp__B1_Correction__N3_Br_20110216143748651_S97012_I217793.nii
Saved skull-stripped image: MRIs_SPLIT_CLASSIFICATION/ADNI_126_S_0865_MR_MPR-R__GradWarp__B1_Correction__N3_Br_20070209205734937_S22295_I39026.nii
Saved skull-stripped image: MRIs_SPLIT_CLASSIFICATION/ADNI_002_S_0685_MR_MPR-R__GradWarp__B1_Correction__N3_Br_20081015130623197_S55756_I120998.nii
Saved skull-stripped image: MRIs_SPLIT_CLASSIFICATION/ADNI_016_S_1121_MR_MPR-R__GradWarp__B1_Correction__N3_Br_20080308123438915_S44322_I96243.nii
Saved skull-stripped image: MRIs_SPLIT_CLASSIFICATION/ADNI_099_S_0090_MR_MPR-R__GradWarp__B1_Correction__N3__Scaled_Br_20080414114559934_S26174_I102436.nii
Saved skull-stripped image: MRIs_SPLIT_CLASSIFICATION/ADNI_022_S_1394_MR_MPR-R__GradWarp__B1_Corre

# assigning classes

In [7]:
import os
import shutil
import pandas as pd
from sklearn.model_selection import train_test_split

def create_classification_dataset(mri_folder, labels_csv, output_folder, test_size=0.1, val_size=0.2, random_state=42):
    """
    Organize MRI images into train/val/test folders for classification,
    mapping numeric labels to meaningful class names.

    Parameters:
        mri_folder (str): Folder containing skull-stripped MRI images
        labels_csv (str): CSV file with columns 'img_name' and 'DIAGNOSIS' (numeric: 1-CN, 2-MCI, 3-AD)
        output_folder (str): Root folder for the classification dataset
        test_size (float): Fraction for test set
        val_size (float): Fraction for validation set
        random_state (int): Random seed
    """

    # Map numeric labels to class names
    label_map = {1: 'CN', 2: 'MCI', 3: 'AD'}

    # Load CSV and drop rows with missing values
    df = pd.read_csv(labels_csv)
    df = df.dropna(subset=['img_name', 'DIAGNOSIS'])

    # Ensure proper types
    df['img_name'] = df['img_name'].astype(str).str.strip()
    df['DIAGNOSIS'] = df['DIAGNOSIS'].astype(int)

    # Create root folders
    for split in ['train', 'val', 'test']:
        os.makedirs(os.path.join(output_folder, split), exist_ok=True)

    # Split into train + temp (val+test), then val/test
    train_df, temp_df = train_test_split(
        df,
        test_size=(test_size + val_size),
        stratify=df['DIAGNOSIS'],
        random_state=random_state
    )
    val_ratio = val_size / (test_size + val_size)
    val_df, test_df = train_test_split(
        temp_df,
        test_size=(1 - val_ratio),
        stratify=temp_df['DIAGNOSIS'],
        random_state=random_state
    )

    splits = {'train': train_df, 'val': val_df, 'test': test_df}

    # Copy images into corresponding folders
    for split_name, split_df in splits.items():
        for _, row in split_df.iterrows():
            class_label = label_map.get(row['DIAGNOSIS'], 'Unknown')
            class_folder = os.path.join(output_folder, split_name, class_label)
            os.makedirs(class_folder, exist_ok=True)

            src = os.path.join(mri_folder, f"{row['img_name']}.nii")
            dst = os.path.join(class_folder, f"{row['img_name']}.nii")

            if os.path.exists(src):
                shutil.copy(src, dst)
            else:
                print(f"Warning: file {src} not found, skipping.")


In [ ]:
create_classification_dataset(
    mri_folder='MRIs_SPLIT_CLASSIFICATION',
    labels_csv='DIAGNOSES_ID_IMGNAME_PATH.csv',
    output_folder='MRIs_CLASSIFICATION',
    test_size=0.1,
    val_size=0.2
)

OSError: [Errno 28] No space left on device: 'MRIs_SPLIT_CLASSIFICATION/ADNI_023_S_0331_MR_MPR-R__GradWarp__B1_Correction__N3_Br_20071226135857055_S19698_I86226.nii' -> 'MRIs_CLASSIFICATION/train/MCI/ADNI_023_S_0331_MR_MPR-R__GradWarp__B1_Correction__N3_Br_20071226135857055_S19698_I86226.nii'

rclone copy /home/e.yavarova/alzheimers_research_lala_emiliya/FASTSURFER_MASKS/ "googledrive:Brain_Segmentation/ADNI-MRI-data/FASTSURFER_MASKS" --drive-shared-with-me -P